In [ ]:
import pandas as pd
from datetime import *
import os 
from scipy.signal import find_peaks as fp


# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES, LABEL_DF

SUPPORTED_FILEFORMATS={"CSV":pd.read_csv,"TXT":pd.read_fwf,"JSON":pd.read_json,"XML":pd.read_xml,"XLSX":pd.read_excel,"XLS":pd.read_excel}
COL_Prefixes ={"HeartRate":"HR_","Metadata":"MD_","WatchAccelerometer":"ACC_", "WatchGravity":"G_","WatchGyroscope":"GYRO_","WatchOrientation":"OR_","WatchTotalAcceleration":"TACC_","WatchMagnetometer":"MAG_"}

RAWDATAFILES={}

MERGE_FILES=1

pd.set_option('display.max_rows', 500)

# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(cwd)) #Parent directory
print("The current working directory is:", WORKING_DIRECTORY)

DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
print("The data directory is:", DATA_DIRECTORY)

RAW_Data_DIR=  os.path.join(DATA_DIRECTORY,"01-raw")
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

#import previous modules output
FEATURES_DF = pd.read_pickle(os.path.join(FEATURES_Data_DIR,"features.pkl"))
EXERCISE_DF = pd.read_pickle(os.path.join(FEATURES_Data_DIR,"exercise_dict.pkl"))


exercise_tuple = [tup for tup in EXERCISE_DF.set_index("key").itertuples()]
print(exercise_tuple)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# packages to create artificial data for clustering purposes
from sklearn.datasets import make_moons, make_blobs


print(FEATURES_DF.columns)
feature_cols = FEATURES_DF.columns[:-3]
target_cols = FEATURES_DF.columns[-3:]

#remove base feature_cols
feature_cols = [ o  for o in feature_cols if ((o.find("peak")>-1 )| (o.find("fft")>-1 )| (o.find("val")>-1 ))]
print(feature_cols)
#limiting the dataset to classes with enough samples:
mask = FEATURES_DF.target.isin([2, 3,  12, 22,25,35])
#mask =((FEATURES_DF.target>1) & (FEATURES_DF.target.isin(list(EXERCISE_DF[EXERCISE_DF.target_count>40].index))))

#38 tricep overhead instead of 19 lat divergent, too similar to pull up
#13 dumbbell press instead of 9 decline abcrunch, to similar to bench pullover


# Separating out the features
x = FEATURES_DF[mask].loc[:, feature_cols].values
y = FEATURES_DF[mask].loc[:,['target']].values


print(x.shape)
print(y.shape)

# Standardizing the features
X = StandardScaler().fit_transform(x)

In [ ]:
# PCA projection to lower dimensional space (number of dimensions depends on n-components parameter)
pcamodel = PCA(n_components=3)
principalComponents = pcamodel.fit_transform(X)
principalDf = pd.DataFrame(data = principalComponents, columns = ['pc1', 'pc2', 'pc3'])
finalDf = pd.concat([principalDf, FEATURES_DF[mask]['target'], FEATURES_DF[mask]['target_name']], axis = 1)


In [ ]:
finalDf.sample(100)

pcamodel2 = PCA(.9)
principalComponents2 = pcamodel2.fit_transform(X)
print(principalComponents2.shape)

In [ ]:
# to define the pallete for graphs
# many different palletes can be chosen https://seaborn.pydata.org/tutorial/color_palettes.html
# sns.set_palette('Paired')
sns.set_palette('Set1') # change the parameter (pallete name) to get different graph appearance

#highest representations:
fig, axs = plt.subplots(2,2, figsize=(20, 10))
sns.scatterplot(x= 'pc1', y='pc2', hue='target', data=finalDf, ax=axs[0,0], alpha=0.5, s=100)
#sns.scatterplot(x= 'pc1', y='pc3', hue='target', data=finalDf, ax=axs[1], alpha=0.5, s=100)
# compare visual results to the previous plot between original feature and target variable
sns.scatterplot(x='mean_OR_pitch', y='fft_TACC_z', hue = 'target_name', data=FEATURES_DF[mask], ax=axs[0,1], alpha=0.5, s=100)
sns.scatterplot(x='mean_OR_pitch', y='mean_peak_TACC_z', hue = 'target_name', data=FEATURES_DF[mask], ax=axs[1,0], alpha=0.5, s=100)
sns.scatterplot(x='mean_OR_pitch', y='fft_TACC_z', hue = 'target_name', data=FEATURES_DF[mask], ax=axs[1,1], alpha=0.5, s=100)
# you can see that applying linear classifier would be a good approach after using PCA

plt.rc('legend', loc="lower right")
axs[0,0].legend(loc="lower right", title="test", title_fontsize=12, fontsize=12)
axs[0,1].legend(loc="lower right", title="test", title_fontsize=12, fontsize=12)
axs[1,0].legend(loc="lower right", title="test", title_fontsize=12, fontsize=12)
sns.set_context("notebook", font_scale=1)

In [ ]:
# plot all 3 axis
# first a bit of aesthetics and colormap change
from matplotlib.colors import ListedColormap
cmap = ListedColormap(sns.color_palette("tab10", 3).as_hex()) # create color map
fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(finalDf['pc1'],finalDf['pc2'],finalDf['pc3'], c = finalDf['target'], cmap=cmap)

In [ ]:
print("low variance ratio")
print((pcamodel.explained_variance_ratio_), sum(pcamodel.explained_variance_ratio_))
print(pcamodel.components_.shape) # has shape [n_components, n_features]


print("39 PC retain 90% variance:")
print((pcamodel2.explained_variance_ratio_), sum(pcamodel2.explained_variance_ratio_)) # 24 PC retain 90% of the variance of original dataset
print(pcamodel2.components_.shape) # has shape [n_components, n_features]


In [ ]:
print("components:")
print(pcamodel.components_)
print("Shape:")
print(pcamodel.components_.shape)

In [ ]:
# How get the graphical overview on how PC are represented by raw features. Hard to see it from previous print results
# fig= plt.figure(figsize=(20, 10))
# fig.dpi = 1024
fig = plt.figure(figsize=(60, 10), dpi=150)
plt.matshow(pcamodel2.components_, cmap='viridis')
plt.yticks([0, 1, 2], ['pc1', 'pc2', 'pc3'])
cbar=plt.colorbar()
#cbar.ax.tick_params(labelsize=2)
plt.xticks(range(len(feature_cols[:10])),feature_cols[:10], rotation=60, ha='left',fontsize=8);
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = [60, 10]
# plt.suptitle("pca_exercises")
plt.show()

In [ ]:
from pca import pca

model = pca(n_components=3)
results = model.fit_transform(X)
# Plot explained variance
fig, ax = model.plot(figsize=(10,5))
# Make biplot with the number of features
fig, ax = model.biplot(n_feat=3, legend=True, figsize=(10,5))
fig, ax = model.biplot3d(n_feat=3, cmap=None, legend=True, figsize=(8,5))
plt.show()